In [2]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import re
from io import StringIO
from bs4 import BeautifulSoup
import contextlib
from selenium.common.exceptions import InvalidArgumentException
import time
import pandas as pd
pd.set_option('display.max_rows', 1000)
import math
import os.path
import pickle
import random

# ignore warnings
pd.options.mode.chained_assignment = None

import undetected_chromedriver as uc

from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium_stealth import stealth

# Log In

In [129]:
driver = uc.Chrome(use_subprocess=True)

In [131]:
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

In [133]:
def mercari_login(driver, username, password):
    # go to mercari login page
    driver.get('https://www.mercari.com/login/')
    
    # # if cookies have been created, load them
    # if os.path.isfile('mercari_cookies.pkl'):
    #     cookies = pickle.load(open("mercari_cookies.pkl", "rb"))
    #     for cookie in cookies:
    #         driver.add_cookie(cookie)
    
    # click "got it" cookies button if it pops up
    try:
        driver.find_element(By.XPATH, '//*[@id="truste-consent-button"]').click()
    except:
        pass

    # enter username and password
    driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/main/div/div/form/div[3]/input').send_keys(username)
    driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/main/div/div/form/div[6]/input').send_keys(password)
    driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/main/div/div/form/div[6]/input').send_keys(Keys.RETURN)

    # click SMS login button if it pops up
    time.sleep(1)
    try:
        sms_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="MFACode"]')))
        sms_button.click()
        # enter SMS code
        sms_code = input('Enter the login code sent via email/SMS: ')
        sms_input = driver.find_element(By.XPATH, '//*[@data-testid="MFACode"]')
        sms_input.send_keys(sms_code)
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="VerifyPhoneMFAButton"]'))).click()
    except:
        pass
    
    # click SMS login button if it pops up
    time.sleep(1)
    try:
        sms_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="VerifyPhoneMFAButton"]')))
        sms_button.click()
        # enter SMS code
        sms_code = input('Enter the login code sent via email/SMS: ')
        sms_input = driver.find_element(By.XPATH, '//*[@data-testid="MFACode"]')
        sms_input.send_keys(sms_code)
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="VerifyPhoneMFAButton"]'))).click()
    except:
        pass
    
    
    driver.get('https://www.mercari.com/')
    time.sleep(2)
    
#     if driver.current_url == 'https://www.mercari.com/login/?login_callback=%2Fmypage%2Flistings%2Factive%2F':
#         mercari_login(driver, username, password)
    
#     # dump cookies
#     pickle.dump(driver.get_cookies(), open("mercari_cookies.pkl", "wb"))


In [135]:
mercari_login(driver, 'username', 'password')

Enter the login code sent via email/SMS:  


In [136]:
pickle.dump(driver.get_cookies(), open("mercari_cookies.pkl", "wb"))

# Get Current Products

In [15]:
def get_current_products(driver):
    
    # define BeautifulSoup as a global variable ("tell Python that the local one is the same as the global one")
    global BeautifulSoup
    
    # go to active listings
    driver.get('https://www.mercari.com/mypage/listings/active/')
    
    # click "got it" cookies button if it pops up
    try:
        driver.find_element(By.XPATH, '//*[@id="truste-consent-button"]').click()
    except:
        pass
    
    # get page source
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')
    
    # get list of products on page
    from bs4 import BeautifulSoup
    import re

    # empty vectors for variable storage
    links = []
    product_names = []
    prices = []
    dates = []

    for i in range(1, 1000):
        try:
            if i==1:
                driver.get('https://www.mercari.com/mypage/listings/active/')
                WebDriverWait(driver, 1.5).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/main/div/div[3]/div[2]/ul/li[1]/div/div[3]/div/div[1]/a')))
            if i>1:
                driver.get('https://www.mercari.com/mypage/listings/active/?page='+str(i))
                WebDriverWait(driver, 1.5).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/main/div/div[3]/div[2]/ul/li[1]/div/div[3]/div/div[1]/a')))
        except:
            pass

        # update page source
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'lxml')

        # if "No results" is present on the page and no products are found, break loop
        if 'No items for sale' in soup.text and len(soup.find_all(attrs={"class":"Link-sc-us5oxt MenuLink-sc-1k5i400 koA-DLz ehDaPW T2-sc-1pugix3 MetaColumn__T2NoWrapEllipsis-sc-18c8425f-1 dNZGZp bhVasx"}, href=True)) == 0:
            print('No more items left, returning variables.')
            time.sleep(1)
            break

        else:
            # get soups
            product_soup = soup.find_all(attrs={"class" : re.compile('ListingsTable__Title')}, href=True)
            price_soup = soup.find_all(attrs={"name": "price"})
            date_soup = soup.find_all("div", {"class": "Row__RowItem-sc-499eeca5-0 hoZBfq"})

            # get product names and links
            for tag in product_soup:
                link = 'https://mercari.com' + tag.get('href')
                links.append(link)
                p_name = str(tag).split('blank">')[-1].split('<')[0]
                product_names.append(p_name)

            # get product prices
            price_soup = soup.find_all(attrs={"id":"price"})
            for tag in price_soup:
                price = float(re.split('value="|"/>', str(tag))[1])
                prices.append(price)

            # get last time products were updated
            date_soup = soup.find_all("td", attrs={"class":"sc-4f08e2f6-0 hoSepO"})
            for tag in date_soup:
                if "/22" in str(tag) or "/23" in str(tag) or "/24" in str(tag) or "/25" in str(tag):
                    date = str(re.split('>|</td>',str(tag))[1])
                    dates.append(date)
    
    return product_names, links, prices, dates
        

In [17]:
product_names, links, prices, dates = get_current_products(driver)

No more items left, returning variables.


In [23]:
for x in [product_names, links, prices, dates]:
    print(len(x))

1172
1172
1172
1172


In [25]:
# dataframe w/ prices and items
dict = {'Item': product_names, 'Link': links, 'Price': prices, 'Last Updated': dates} 

df = pd.DataFrame(dict)
display(df)

,Item,Link,Price,Last Updated
0,Shonen Jump Issue 15 April 2004 Yu Gi OH Cover...,https://mercari.com/us/item/m65957477729/,6.99,12/31/24
1,Liquid Death Don't F The Planet Death To Plast...,https://mercari.com/us/item/m86523504539/,9.99,12/30/24
2,Sony RMT-D197A DVD Player Remote Control Teste...,https://mercari.com/us/item/m71278368472/,11.99,12/30/24
3,Sony RM-ADU007 AV System Remote Control Tested...,https://mercari.com/us/item/m98037395870/,9.99,12/30/24
4,Wincraft ESPN U Hanging Plastic Sign 11x17 Alw...,https://mercari.com/us/item/m84785797973/,18.99,12/29/24
...,...,...,...,...
1167,Vintage Street Rodder Magazine March 1993 Volu...,https://mercari.com/us/item/m64574459359/,6.22,08/12/24
1168,Vintage Champion Logo T Shirt Blue Bar Size M ...,https://mercari.com/us/item/m11259306723/,7.20,08/12/24
1169,Bridgestone Firestone Hat Hot Graphic Black Au...,https://mercari.com/us/item/m12440616746/,6.92,08/12/24
1170,Vintage Hawaii Souvenir Hat Black Pink Snap Ba...,https://mercari.com/us/item/m72035723594/,7.06,08/12/24


# Identify Listings That Were Sold on Vendoo So They Can Be Deleted From Mercari

In [27]:
active_df = df.copy()

In [29]:
active_df

,Item,Link,Price,Last Updated
0,Shonen Jump Issue 15 April 2004 Yu Gi OH Cover...,https://mercari.com/us/item/m65957477729/,6.99,12/31/24
1,Liquid Death Don't F The Planet Death To Plast...,https://mercari.com/us/item/m86523504539/,9.99,12/30/24
2,Sony RMT-D197A DVD Player Remote Control Teste...,https://mercari.com/us/item/m71278368472/,11.99,12/30/24
3,Sony RM-ADU007 AV System Remote Control Tested...,https://mercari.com/us/item/m98037395870/,9.99,12/30/24
4,Wincraft ESPN U Hanging Plastic Sign 11x17 Alw...,https://mercari.com/us/item/m84785797973/,18.99,12/29/24
...,...,...,...,...
1167,Vintage Street Rodder Magazine March 1993 Volu...,https://mercari.com/us/item/m64574459359/,6.22,08/12/24
1168,Vintage Champion Logo T Shirt Blue Bar Size M ...,https://mercari.com/us/item/m11259306723/,7.20,08/12/24
1169,Bridgestone Firestone Hat Hot Graphic Black Au...,https://mercari.com/us/item/m12440616746/,6.92,08/12/24
1170,Vintage Hawaii Souvenir Hat Black Pink Snap Ba...,https://mercari.com/us/item/m72035723594/,7.06,08/12/24


In [31]:
driver.get('https://web.vendoo.co/login/')

In [33]:
# define username and password
username='username'
password='password'

# enter username and password
driver.find_element(By.XPATH, '//*[@id="email"]').send_keys(username)

driver.find_element(By.XPATH, '//*[@id="password"]').send_keys(password)
driver.find_element(By.XPATH, '//*[@id="password"]').send_keys(Keys.RETURN)

In [93]:
driver.get('https://web.vendoo.co/app?status=sold')

In [39]:
# clear sold listing detections
from selenium.webdriver.common.action_chains import ActionChains

try:
    element = driver.find_element(By.XPATH, '//*[@aria-label="Clear All Latest Detection"]')
    actions = ActionChains(driver)
    actions.move_to_element(element)
    actions.perform()
    driver.execute_script('window.scrollBy(0, 100)')
    element.click()
except:
    pass

In [95]:
# scroll down to bottom while allowing time for products to load V2
curr_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
for i in range(0,1000):
    page = driver.find_element("tag name","html")
    page.send_keys(Keys.END)
    time.sleep(0.5)
    new_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
    try:
        driver.find_element(By.XPATH, "/html/body/div[1]/div/div[1]/div/div[3]/div/div/div[1]/div[1]/div[2]/div/div/div[2]/div[1]/div/div[3]/div/div/span/div[2]/div/div")
    except:
        break

In [97]:
# change to 240 items/page
driver.find_element(By.XPATH, '//*[@class="buttons__ButtonDefaultStyled-rvbg1v-10 styles__ItemsPerPageButton-wgdgzp-1 hBjREI buqMst"]').click()
driver.find_element(By.CSS_SELECTOR, '.EESAW > div:nth-child(4)').click()
# driver.find_element(By.XPATH, '//*[@data-testid="dropdown__option__240"]').click()

In [99]:
# scroll down to bottom while allowing time for products to load V2
curr_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
for i in range(0,1000):
    page = driver.find_element("tag name","html")
    page.send_keys(Keys.END)
    time.sleep(0.5)
    new_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
    try:
        driver.find_element(By.XPATH, "/html/body/div[1]/div/div[1]/div/div[3]/div/div/div[1]/div[1]/div[2]/div/div/div[2]/div[1]/div/div[3]/div/div/span/div[2]/div/div")
    except:
        break

In [101]:
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

In [103]:
vendoo_df = pd.read_csv('vendoo_sold_products.csv')

In [105]:
vendoo_list = vendoo_df['product'].tolist()

In [117]:
# get sold products from Vendoo
sold_products = []
num_pages = int(re.split('\n', driver.find_element(By.CLASS_NAME, 'rc-pagination').text)[-1])
for i in range(num_pages):
    # scroll to top
    page = driver.find_element("tag name","html")
    page.send_keys(Keys.HOME)
    
    time.sleep(1)
    
    # scroll down
    curr_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
    for i in range(0,100):
        page = driver.find_element("tag name","html")
        page.send_keys(Keys.END)
        time.sleep(0.5)
        new_scrollheight = driver.execute_script("return document.documentElement.scrollHeight")
        if new_scrollheight != curr_scrollheight:
            curr_scrollheight = new_scrollheight
        else:
            break
    
    # wait
    time.sleep(1)
    
    # get products
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')
    page_product_soup = soup.find_all("span",attrs={"class":"styles__HighlightTextStyled-jahwfy-5 fvIvfJ"})
    for tag in page_product_soup:
        # print(tag.text)
        sold_products.append(tag.text)
        if tag.text not in vendoo_list:
            vendoo_list.insert(0, tag.text)
        else:
            print('item already in list of sold items')
            break
    else:
        continue
        print(sold_products[-1],str(len(sold_products)), '\n')
        page.send_keys(Keys.END)
        r_chevron = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="chevron-right"]')))
        r_chevron.click()
        time.sleep(1)
    break

item already in list of sold items


In [119]:
vendoo_df = pd.DataFrame({'product': vendoo_list})

In [123]:
vendoo_df.to_csv('vendoo_sold_products.csv', index=False)

In [127]:
# determine products that are marked as sold on Vendoo but are still active on Mercari
soldlist = []
for i in range(len(active_df)):
    product_name = active_df.iloc[i,0]
    if product_name in vendoo_list:
        soldlist.append(product_name)
        
soldlist_df = pd.DataFrame({'Item':soldlist})
soldlist_df.insert(loc = 1,column = 'Deleted',value = '<input type="checkbox" />')
soldlist_df.style

,Item,Deleted
0,Wincraft ESPN U Hanging Plastic Sign 11x17 Always Loud Never Graduate,
1,Akai HX-1 Stereo Cassette Deck Tested Working Belts Replaced Internals Cleaned,
2,LEGO Batman 2 DC Super Heroes Microsoft Xbox 360 Tested Working Complete CIB,
3,OEM Microsoft Xbox 360 Wireless Controller Tested Working Black,
4,Kinect Sports Season Two 2 Xbox 360 2011 Complete CIB Tested Working,
5,Signatures Boston College Eagles Visor Hat Adjustable Red Gold Embroidered,
6,LEGO Batman / Pure 2 Pack Microsoft Xbox 360 Tested Working Complete CIB,
7,VTG Jim Beam Sleeveless Shirt Size XL Throw Down The Rock Basketball Promo,
8,Urban Outfitters Eat My Dust Crewneck Sweatshirt Size L/XL Racing Graphic,
9,Vintage 1979 Toytronic Electronic Tabletop Pinball Game Fully Tested Space Ships,


# Get Products Again After Deleting Sold Listings

In [152]:
product_names, links, prices, dates = get_current_products(driver)

No more items left, returning variables.


In [154]:
for x in [product_names, links, prices, dates]:
    print(len(x))

1181
1181
1181
1181


In [156]:
# dataframe w/ prices and items
dict = {'Item': product_names, 'Link': links, 'Price': prices, 'Last Updated': dates} 

df = pd.DataFrame(dict)
display(df)

,Item,Link,Price,Last Updated
0,Shonen Jump Issue 15 April 2004 Yu Gi OH Cover...,https://mercari.com/us/item/m65957477729/,6.99,12/31/24
1,Liquid Death Don't F The Planet Death To Plast...,https://mercari.com/us/item/m86523504539/,9.99,12/30/24
2,Sony RMT-D197A DVD Player Remote Control Teste...,https://mercari.com/us/item/m71278368472/,11.99,12/30/24
3,Sony RM-ADU007 AV System Remote Control Tested...,https://mercari.com/us/item/m98037395870/,9.99,12/30/24
4,Wincraft ESPN U Hanging Plastic Sign 11x17 Alw...,https://mercari.com/us/item/m84785797973/,18.99,12/29/24
...,...,...,...,...
1176,Vintage Street Rodder Magazine March 1993 Volu...,https://mercari.com/us/item/m64574459359/,6.22,08/12/24
1177,Vintage Champion Logo T Shirt Blue Bar Size M ...,https://mercari.com/us/item/m11259306723/,7.20,08/12/24
1178,Bridgestone Firestone Hat Hot Graphic Black Au...,https://mercari.com/us/item/m12440616746/,6.92,08/12/24
1179,Vintage Hawaii Souvenir Hat Black Pink Snap Ba...,https://mercari.com/us/item/m72035723594/,7.06,08/12/24


# Drop Price By a Percentage

In [159]:
from datetime import datetime
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
df['Days Since Updating'] = (datetime.today()-df['Last Updated']).dt.days
display(df)

C:\Users\babow\AppData\Local\Temp\ipykernel_23396\4140236999.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Last Updated'] = pd.to_datetime(df['Last Updated'])


,Item,Link,Price,Last Updated,Days Since Updating
0,Shonen Jump Issue 15 April 2004 Yu Gi OH Cover...,https://mercari.com/us/item/m65957477729/,6.99,2024-12-31,3
1,Liquid Death Don't F The Planet Death To Plast...,https://mercari.com/us/item/m86523504539/,9.99,2024-12-30,4
2,Sony RMT-D197A DVD Player Remote Control Teste...,https://mercari.com/us/item/m71278368472/,11.99,2024-12-30,4
3,Sony RM-ADU007 AV System Remote Control Tested...,https://mercari.com/us/item/m98037395870/,9.99,2024-12-30,4
4,Wincraft ESPN U Hanging Plastic Sign 11x17 Alw...,https://mercari.com/us/item/m84785797973/,18.99,2024-12-29,5
...,...,...,...,...,...
1176,Vintage Street Rodder Magazine March 1993 Volu...,https://mercari.com/us/item/m64574459359/,6.22,2024-08-12,144
1177,Vintage Champion Logo T Shirt Blue Bar Size M ...,https://mercari.com/us/item/m11259306723/,7.20,2024-08-12,144
1178,Bridgestone Firestone Hat Hot Graphic Black Au...,https://mercari.com/us/item/m12440616746/,6.92,2024-08-12,144
1179,Vintage Hawaii Souvenir Hat Black Pink Snap Ba...,https://mercari.com/us/item/m72035723594/,7.06,2024-08-12,144


In [161]:
# get rid of recently updated listings
pricedrop_df = df.loc[(df['Days Since Updating'] >= 7) ].reset_index(drop=True)

# sort by price, descending
pricedrop_df = pricedrop_df.sort_values('Price', ascending = False).reset_index(drop=True)

# display dataframe of products to de price dropped
display(pricedrop_df.loc[(pricedrop_df['Price'] > 10) ].reset_index(drop=True))

,Item,Link,Price,Last Updated,Days Since Updating
0,Pair of 2 Technics SB-K25 Floor Speakers 3-Way...,https://mercari.com/us/item/m99784160116/,342.94,2024-12-11,23
1,Panasonic RF-2800 Shortwave AM FM Radio Tested...,https://mercari.com/us/item/m82056945871/,299.99,2024-12-27,7
2,Yamaha R-N301 Stereo Receiver Fully Tested No ...,https://mercari.com/us/item/m13560056978/,249.99,2024-12-27,7
3,Bose 321 GS Series II DVD Home Entertainment S...,https://mercari.com/us/item/m99939566800/,199.99,2024-12-23,11
4,Vintage Claudia Grau Patchwork Blazer Jacket S...,https://mercari.com/us/item/m78376995345/,180.49,2024-12-11,23
5,Set of 2 EPI T/E 70 Speakers Time Energy Serie...,https://mercari.com/us/item/m16213763349/,171.00,2024-12-11,23
6,Vintage Zenith C585W Turntable AM FM Radio Tes...,https://mercari.com/us/item/m81180216486/,169.99,2024-12-19,15
7,Life Like Racing NASCAR Electric Slot Car Cham...,https://mercari.com/us/item/m57823475027/,149.99,2024-12-12,22
8,Royal Retros Vintage Icons Washington Senators...,https://mercari.com/us/item/m81077460745/,126.57,2024-12-11,23
9,Vintage Sawyers Mirascreen Slide Projector Vie...,https://mercari.com/us/item/m21110171190/,99.99,2024-12-10,24


In [163]:
def drop_prices(driver, input_df, percent_discount, floor_price):
    input_df = input_df.loc[(input_df['Price'] > floor_price) ].reset_index(drop=True)
    print('Number of items to be discounted:', str(len(input_df)), '\n')
    display(input_df)
    for i in range(0, len(input_df)):
        item = input_df.iloc[i,list(input_df.columns).index('Item')]
        link = input_df.iloc[i,list(input_df.columns).index('Link')]
        curr_price = input_df.iloc[i,list(input_df.columns).index('Price')]
        reduced_price = curr_price - (curr_price * (percent_discount*0.01))
        reduced_price = round(reduced_price, 2)
        # don't drop prices below floor_price
        if reduced_price < floor_price:
            reduced_price = floor_price

        # don't waste time on products already at the floor price    
        if curr_price == reduced_price:
            print('Item '+ str(i+1) + ' of ' + str(len(input_df)), '('+item+')\n', 'Old Price: '+ str(curr_price), 'New Price: '+ str(reduced_price))
            print('\tItem already at floor price.')

        elif curr_price < reduced_price:
            print('Item '+ str(i+1) + ' of ' + str(len(input_df)), '('+item+')\n', 'Old Price: '+ str(curr_price), 'New Price: '+ str(reduced_price))
            print('\tItem already priced below floor price.')

        else:
            print('Item '+ str(i+1) + ' of ' + str(len(input_df)), '('+item+')\n', 'Old Price: '+ str(curr_price), 'New Price: '+ str(reduced_price))
            driver.get(link)
            # click "got it" cookies button if it pops up
            try:
                driver.find_element(By.XPATH, '//*[@id="truste-consent-button"]').click()
            except:
                pass
            time.sleep(1)
            price_element = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="price"]')))
            price_element.click()
            price_element.send_keys(Keys.CONTROL + "a");
            price_element.send_keys(Keys.DELETE);
            price_element.send_keys(str(reduced_price))
            price_element.send_keys(Keys.RETURN);
            time.sleep(3)

    print('Done.')
    driver.quit()

In [165]:
# drop_prices(driver, input_df, percent_discount, floor_price)
drop_prices(driver, pricedrop_df, 5, 10)

Number of items to be discounted: 551 



,Item,Link,Price,Last Updated,Days Since Updating
0,Pair of 2 Technics SB-K25 Floor Speakers 3-Way...,https://mercari.com/us/item/m99784160116/,342.94,2024-12-11,23
1,Panasonic RF-2800 Shortwave AM FM Radio Tested...,https://mercari.com/us/item/m82056945871/,299.99,2024-12-27,7
2,Yamaha R-N301 Stereo Receiver Fully Tested No ...,https://mercari.com/us/item/m13560056978/,249.99,2024-12-27,7
3,Bose 321 GS Series II DVD Home Entertainment S...,https://mercari.com/us/item/m99939566800/,199.99,2024-12-23,11
4,Vintage Claudia Grau Patchwork Blazer Jacket S...,https://mercari.com/us/item/m78376995345/,180.49,2024-12-11,23
5,Set of 2 EPI T/E 70 Speakers Time Energy Serie...,https://mercari.com/us/item/m16213763349/,171.00,2024-12-11,23
6,Vintage Zenith C585W Turntable AM FM Radio Tes...,https://mercari.com/us/item/m81180216486/,169.99,2024-12-19,15
7,Life Like Racing NASCAR Electric Slot Car Cham...,https://mercari.com/us/item/m57823475027/,149.99,2024-12-12,22
8,Royal Retros Vintage Icons Washington Senators...,https://mercari.com/us/item/m81077460745/,126.57,2024-12-11,23
9,Vintage Sawyers Mirascreen Slide Projector Vie...,https://mercari.com/us/item/m21110171190/,99.99,2024-12-10,24


Item 1 of 551 (Pair of 2 Technics SB-K25 Floor Speakers 3-Way System Tested Working No Scuffs)
 Old Price: 342.94 New Price: 325.79
Item 2 of 551 (Panasonic RF-2800 Shortwave AM FM Radio Tested Working w/ Power Cord and Antenna)
 Old Price: 299.99 New Price: 284.99
Item 3 of 551 (Yamaha R-N301 Stereo Receiver Fully Tested No Remote Net Connected 2 Channel)
 Old Price: 249.99 New Price: 237.49
Item 4 of 551 (Bose 321 GS Series II DVD Home Entertainment System Complete Fully Tested Works)
 Old Price: 199.99 New Price: 189.99
Item 5 of 551 (Vintage Claudia Grau Patchwork Blazer Jacket Size Petite 4 Bespoke 80s)
 Old Price: 180.49 New Price: 171.47
Item 6 of 551 (Set of 2 EPI T/E 70 Speakers Time Energy Series Fully Tested Working New Rings)
 Old Price: 171.0 New Price: 162.45
Item 7 of 551 (Vintage Zenith C585W Turntable AM FM Radio Tested Working Wood Grain New Stylus)
 Old Price: 169.99 New Price: 161.49
Item 8 of 551 (Life Like Racing NASCAR Electric Slot Car Championship Challenge 99%

WebDriverException: Message: target frame detached: received Inspector.detached event
  (Session info: chrome=131.0.6778.205)
Stacktrace:
	GetHandleVerifier [0x0064EC13+23731]
	(No symbol) [0x005DC394]
	(No symbol) [0x004BBE63]
	(No symbol) [0x004AE9B8]
	(No symbol) [0x004ADBC7]
	(No symbol) [0x004AD527]
	(No symbol) [0x004AD441]
	(No symbol) [0x004AB8DB]
	(No symbol) [0x004ABF4D]
	(No symbol) [0x004B81EA]
	(No symbol) [0x004C78A5]
	(No symbol) [0x004CC6F6]
	(No symbol) [0x004AC555]
	(No symbol) [0x004C74D1]
	(No symbol) [0x0053B7A1]
	(No symbol) [0x00521BF6]
	(No symbol) [0x004F3F35]
	(No symbol) [0x004F4EBD]
	GetHandleVerifier [0x0092F0D3+3039603]
	GetHandleVerifier [0x00942DEA+3120778]
	GetHandleVerifier [0x0093B592+3089970]
	GetHandleVerifier [0x006E43B0+635984]
	(No symbol) [0x005E4DCD]
	(No symbol) [0x005E2068]
	(No symbol) [0x005E2205]
	(No symbol) [0x005D4FD0]
	BaseThreadInitThunk [0x7645FCC9+25]
	RtlGetAppContainerNamedObjectPath [0x77D9809E+286]
	RtlGetAppContainerNamedObject

In [ ]:
pricedrop_df

# Automate Sending Offers (XPATHs Here Likely Need To Be Updated)

In [70]:
driver = start_driver()

In [69]:
mercari_login(driver, 'username', 'password')

In [53]:
# go to active listings, sorted by number of likes
driver.get('https://www.mercari.com/mypage/listings/active/?sortBy=3')

In [54]:
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

In [83]:
def send_offers(driver, discount_pct, floor_price):
    global BeautifulSoup
    
    # go to active listings, sorted by number of likes
    driver.get('https://www.mercari.com/mypage/listings/active/?sortBy=3')

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'lxml')

    # get list of products on page
    from bs4 import BeautifulSoup
    import re

    # empty vectors for variable storage
    links = []
    product_names = []
    prices = []
    dates = []

    for i in range(0, 1000):
        try:
            if i==1:
                driver.get('https://www.mercari.com/mypage/listings/active/?sortBy=3')
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/main/div/div[3]/div[2]/ul/li[1]/div/div[3]/div/div[1]/a')))
            if i>1:
                driver.get('https://www.mercari.com/mypage/listings/active/?sortBy=3&page='+str(i))
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[4]/main/div/div[3]/div[2]/ul/li[1]/div/div[3]/div/div[1]/a')))
        except:
            pass

        # update page source
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'lxml')

        # if "No results" is present on the page and no products are found, break loop
        if 'No items for sale' in soup.text and len(soup.find_all(attrs={"class":"Link-sc-us5oxt MenuLink-sc-1k5i400 koA-DLz ehDaPW T2-sc-1pugix3 MetaColumn__T2NoWrapEllipsis-sc-18c8425f-1 dNZGZp bhVasx"}, href=True)) == 0:
            print('No more items, done.')
            break

        else:
            # get item names, prices, offer button elements, number of likers
            itemnames = [x.text for x in driver.find_elements(By.XPATH, '//*[@data-testid="ItemLink"]')]
            prices = [float(x['value']) for x in soup.find_all("input", attrs={"name":"price"})]
            num_likers = [int(tag.text) for tag in driver.find_elements(By.XPATH, '//*[@class="T2-sc-1um8956 daMJEj"]')[1::2]]

            for i in range(len(itemnames)):
                itemname = itemnames[i]
                itemlikes = num_likers[i]
                itemprice = prices[i]

                if itemlikes > 0:
                    time.sleep(1)
                    offer_button = WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, '//*[@data-testid="OfferButton"]')))[i]
                    driver.execute_script("arguments[0].scrollIntoView(true);", offer_button)
                    driver.execute_script("window.scrollBy(0, -400);")
                    rev_discount = 1 - (0.01 * discount_pct)
                    reduced_price = int(math.floor(rev_discount * itemprice))
                    if itemprice <= floor_price:
                        print(itemname, 'is below the provided floor price. No offer sent.')
                        pass
                    else:
                        offer_button.click()

                        # get suggested reduced price for invalid offer cases
                        time.sleep(1)
                        suggested_discountprice = float(WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="OfferToLikerInput"]'))).get_attribute('value'))
                        time.sleep(0.5)
                        if reduced_price > suggested_discountprice:
                            price_element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@name="offerPrice"]')))
                            price_element.click()
                            price_element.send_keys(Keys.RETURN);
                        else: 
                            price_element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@name="offerPrice"]')))
                            price_element.click()
                            price_element.send_keys(Keys.CONTROL + "a");
                            price_element.send_keys(Keys.DELETE);
                            price_element.send_keys(str(reduced_price))
                            price_element.send_keys(Keys.RETURN);
                        time.sleep(2)

                        # if an offer has already been sent, don't send
                        try:
                            # if there is an error when offering to a liker
                            if driver.find_element(By.XPATH, '//*[@data-testid="OfferToLikerError"]'):
                                # cases where an offer needs to be lower
                                if 'needs' in driver.find_element(By.XPATH, '//*[@data-testid="OfferToLikerError"]').text:
                                    highest_possible_offerprice = float(re.split('\$', driver.find_element(By.XPATH, '//*[@data-testid="OfferToLikerError"]').text)[-1])
                                    price_element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@name="offerPrice"]')))
                                    price_element.click()
                                    price_element.send_keys(Keys.CONTROL + "a")
                                    price_element.send_keys(Keys.DELETE)
                                    price_element.send_keys(str(highest_possible_offerprice))
                                    price_element.send_keys(Keys.RETURN)
                                    sendoffer_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="MakeAnOfferButton"]')))
                                    sendoffer_button.click()
                                    print('Offer sent for', itemname + '.')
                                    time.sleep(2)
                                    driver.refresh()
                                    time.sleep(2.5)
                                # cases where an offer has already been sent
                                else:
                                    ActionChains(driver).send_keys(Keys.ESCAPE).perform()
                                    print('Offer has already been sent for', itemname + ' within the past 24 hours.')
                        # otherwise, send offer
                        except:
                            sendoffer_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@data-testid="MakeAnOfferButton"]')))
                            sendoffer_button.click()
                            print('Offer sent for', itemname + '.')
                            time.sleep(2.5)
                            driver.refresh()
                            time.sleep(2.5)
                else:
                    print('No items with likers on this page.')
                    break

In [84]:
# change i to start at page 5, try again. last error resulted from what mercari says is an item with no likes but it does have likes

In [85]:
send_offers(driver, 15, 10)

Adult Swim Metalocalypse Season 2 DVD Set Two Discs Tested Working is below the provided floor price. No offer sent.
Spongebob Squarepants Laugh Your Pants Off VHS Tested Working 5 Episodes 2003 is below the provided floor price. No offer sent.
Offer sent for Verit Industries Model 1030 Speakers Vintage Home Audio Tested Working 70s.
Offer sent for Element Skateboards All Terrain Urethane Skateboard Wheels 50mm Black Light Blue.
Offer sent for VTG Sony STR-VX1 Stereo Receiver AM FM Phono Tape Inputs Fully Tested Working.
Offer sent for Disney Toy Story Pizza Planet Sweatshirt Size L Black Red Tie Dye.
Offer sent for Fox Racing Flex Fit Baseball Hat Size L/XL Black Green White.
Offer sent for Sony Playstation 2 PS2 Dualshock Controller Clear Smoke Gray SCPH-10010 Tested.
Offer sent for Vintage The Mountain Wolf Shirt Mens Sz XL Tie Dye SS Native American Art Tee.
Offer sent for VTG 90s Pokemon Gotta Catch Em All Charmander Card Binder Holds 136 Cards Empty.
Offer sent for Shimano Ultegr

Offer sent for Atomic Betaflex Ski Boots Size Mens 9.5 Womens 10.5 316mm Black.
Offer sent for Lot of 16 White Dwarf Magazines 2007 2018 2019 Games Workshop Warhammer 40K.
Offer sent for Lenovo ThinkCentre TIO24D 24" Tiny-In-One Widescreen LED IPS Monitor Tested.
Offer sent for Barbie Golfer T Shirt Size L Golf Pink Unisex Graphic Tee.
Offer sent for The Twilight Zone 2 Walking Distance Audio Book Cassette Rod Serling.
Offer sent for VTG New Era 2003 World Series 100th Anniversary Hat Yankees Marlins Reshaped.
Offer sent for Encore Tape 2 USB Cassette Converter Model 2034 Tested Working.
Angels And Airwaves We Dont Need To Whisper CD 2006 Tested Working is below the provided floor price. No offer sent.
Vintage New Era MLB All Star Fanfest Boston 1999 Snapback Hat Cleaned Reshaped is below the provided floor price. No offer sent.
Offer sent for VTG Smith Corona Electra CT Portable Electric Typewriter With Case Tested Blue.
Offer sent for Vintage EZ's By Haggar Full Zip Windbreaker Jacke

TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF608FDEEA2+31554]
	(No symbol) [0x00007FF608F57ED9]
	(No symbol) [0x00007FF608E1872A]
	(No symbol) [0x00007FF608E68434]
	(No symbol) [0x00007FF608E6853C]
	(No symbol) [0x00007FF608EAF6A7]
	(No symbol) [0x00007FF608E8D06F]
	(No symbol) [0x00007FF608EAC977]
	(No symbol) [0x00007FF608E8CDD3]
	(No symbol) [0x00007FF608E5A33B]
	(No symbol) [0x00007FF608E5AED1]
	GetHandleVerifier [0x00007FF6092E8B1D+3217341]
	GetHandleVerifier [0x00007FF609335AE3+3532675]
	GetHandleVerifier [0x00007FF60932B0E0+3489152]
	GetHandleVerifier [0x00007FF60908E776+750614]
	(No symbol) [0x00007FF608F6375F]
	(No symbol) [0x00007FF608F5EB14]
	(No symbol) [0x00007FF608F5ECA2]
	(No symbol) [0x00007FF608F4E16F]
	BaseThreadInitThunk [0x00007FFADD7A7374+20]
	RtlUserThreadStart [0x00007FFADDF1CC91+33]
